# Step 6 — Daily branch transaction forecasting dataset

**Scope:** construct and audit features only. No forecasting model is trained, no imputation is performed, and no train/test split is selected here. Target = observed daily unique transactions for each branch. Recorded sales are not unconstrained customer demand.

Forecast origin: after the previous day closes, for the next calendar day. Lag 1 cannot be reused unchanged for forecasts issued a week ahead.

In [1]:
from pathlib import Path
import os, sys, json, hashlib, inspect
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT/'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))
from prepare_daily_demand import build_daily_features, save_features, FEATURES
DATA = Path(os.environ.get('STORIES_DATA', '/Users/aya/Downloads/stories_transactions.csv'))
raw = pd.read_csv(DATA)
print('Raw line items:', len(raw))
print('Unique transaction IDs:', raw['Transaction ID'].nunique())

Raw line items: 51177
Unique transaction IDs: 33887


## 1. Calendar-complete panel and missing branch-days

Include every combination of Garden, Harbour and Uptown with January 1–June 30, 2026. Repeated basket lines count once. Absent branch-days remain unknown (`NA`), not zero; a lack of transactions cannot distinguish closure, a missing extract or genuine zero sales. The input is checked for invalid headers and conflicting branch/date information within transaction IDs.

In [2]:
daily = build_daily_features(raw)
missing_branch_days = daily.loc[daily['MissingBranchDay'], ['Branch','Date']]
print('Total observations:', len(daily))
print('Missing branch-days:', len(missing_branch_days))
display(missing_branch_days)
display(daily.groupby('Branch').agg(calendar_dates=('Date','size'),observed_dates=('Transactions','count')))

Total observations: 543
Missing branch-days: 0


,Branch,Date


,calendar_dates,observed_dates
Branch,,
Garden,181,181
Harbour,181,181
Uptown,181,181


## 2. Features and information availability

Calendar variables are known in advance. `DayOfWeek` is a name; `Weekend` is Saturday/Sunday = 1. Target is `Transactions`. `MissingBranchDay` is an audit flag, not a predictor.

Within each branch, sorted by the dense calendar: `Lag1 = Transactions.shift(1)`, `Lag7 = Transactions.shift(7)`, and `RollingMean7 = Transactions.shift(1).rolling(7, min_periods=7).mean()`. Seven **calendar** dates are required; missing observations propagate. No backward fill, same-day target input or cross-branch shift is used.

In [3]:
print(inspect.getsource(build_daily_features))
print('Missing values in requested features:')
display(daily[FEATURES].isna().sum().rename('Missing values').to_frame())
print('Missing target values:', daily['Transactions'].isna().sum())
print('Missing values in all exported columns:')
display(daily.isna().sum().rename('Missing values').to_frame())

def build_daily_features(raw):
 required=['Transaction ID','Date','Branch']
 if not set(required)<=set(raw):raise ValueError(f'Required columns: {required}')
 if raw[required].isna().any().any():raise ValueError('Missing transaction IDs, branch labels or dates need investigation')
 x=raw[required].copy();x['Date']=pd.to_datetime(x.Date,format='%Y-%m-%d',errors='raise')
 if not set(x.Branch)<=set(BRANCHES):raise ValueError('Unexpected branch; revise canonical branch list')
 if x.groupby('Transaction ID')[['Date','Branch']].nunique().gt(1).any().any():raise ValueError('A transaction ID has conflicting branch/date headers')
 x=x[x.Date.between(START,END)]
 observed=x.groupby(['Branch','Date'])['Transaction ID'].nunique().rename('Transactions')
 grid=pd.MultiIndex.from_product([BRANCHES,pd.date_range(START,END)],names=['Branch','Date'])
 # Missing rows stay unknown. No fillna(0), backward filling or current-day target input.
 daily=observed.reindex(grid).reset_index();daily['Transactions']

,Missing values
Branch,0
DayOfWeek,0
Month,0
DayOfMonth,0
Weekend,0
Lag1,3
Lag7,21
RollingMean7,21


,Missing values
Branch,0
Date,0
DayOfWeek,0
Month,0
DayOfMonth,0
Weekend,0
Lag1,3
Lag7,21
RollingMean7,21
Transactions,0


## 3. First 15 observations

Sorted by branch and date. Initial lag/rolling blanks are warm-up history, not missing branch-days. Keep these rows in the saved dataset; how to handle warm-up rows belongs to the later evaluation protocol.

In [4]:
display(daily.head(15))
print('Rows with complete target and requested features:', daily[FEATURES+['Transactions']].notna().all(axis=1).sum())

Rows with complete target and requested features: 522


,Branch,Date,DayOfWeek,Month,DayOfMonth,Weekend,Lag1,Lag7,RollingMean7,Transactions,MissingBranchDay
0,Garden,2026-01-01,Thursday,1,1,0,<NA>,<NA>,NaN,48,False
1,Garden,2026-01-02,Friday,1,2,0,48,<NA>,NaN,48,False
2,Garden,2026-01-03,Saturday,1,3,1,48,<NA>,NaN,36,False
3,Garden,2026-01-04,Sunday,1,4,1,36,<NA>,NaN,47,False
4,Garden,2026-01-05,Monday,1,5,0,47,<NA>,NaN,32,False
5,Garden,2026-01-06,Tuesday,1,6,0,32,<NA>,NaN,73,False
6,Garden,2026-01-07,Wednesday,1,7,0,73,<NA>,NaN,40,False
7,Garden,2026-01-08,Thursday,1,8,0,40,48,46.285714,37,False
8,Garden,2026-01-09,Friday,1,9,0,37,48,44.714286,50,False
9,Garden,2026-01-10,Saturday,1,10,1,50,36,45.000000,43,False


## 4. Verify temporal isolation and handling of a missing day

Deleting observations on and after June 1 must not alter June 1 or earlier predictor values. Separately remove Garden January 20: its target stays unknown, January 21 lag 1 is unknown, January 27 lag 7 is unknown, and prior-seven-day means remain unknown while that missing date is in the window. These are deliberately modified copies for checks; the original CSV is unchanged.

In [5]:
cutoff = pd.Timestamp('2026-06-01')
past_only = raw[pd.to_datetime(raw['Date']) < cutoff].copy()
modified = build_daily_features(past_only)
pd.testing.assert_frame_equal(daily.loc[daily.Date<=cutoff, ['Date']+FEATURES], modified.loc[modified.Date<=cutoff, ['Date']+FEATURES])
gap_raw = raw[~((raw['Branch']=='Garden') & (raw['Date']=='2026-01-20'))].copy()
gap = build_daily_features(gap_raw).set_index(['Branch','Date'])
assert pd.isna(gap.loc[('Garden',pd.Timestamp('2026-01-20')), 'Transactions'])
assert pd.isna(gap.loc[('Garden',pd.Timestamp('2026-01-21')), 'Lag1'])
assert pd.isna(gap.loc[('Garden',pd.Timestamp('2026-01-27')), 'Lag7'])
for day in pd.date_range('2026-01-21','2026-01-27'):
    assert pd.isna(gap.loc[('Garden',day),'RollingMean7'])
assert pd.notna(gap.loc[('Garden',pd.Timestamp('2026-01-28')), 'RollingMean7'])
assert daily.Transactions.sum() == raw['Transaction ID'].nunique()
assert not daily.duplicated(['Branch','Date']).any()
print('PASS temporal isolation, missing-day propagation, unique keys, transaction conservation.')

PASS temporal isolation, missing-day propagation, unique keys, transaction conservation.


## 5. Save the forecasting dataset

Output: `daily_demand_features.csv` in the repository root, with missing numeric values as empty CSV fields. Missing-branch-day list and feature summary go to `reports/demand_forecasting/`. The CSV contains derived daily operational data and stays local by default. No customer IDs, product lines or transaction IDs are exported. There are no model results in this step.

In [6]:
summary = save_features(raw, daily, ROOT/'daily_demand_features.csv', ROOT/'reports/demand_forecasting', hashlib.sha256(DATA.read_bytes()).hexdigest())
round_trip = pd.read_csv(ROOT/'daily_demand_features.csv')
assert len(round_trip) == len(daily)
assert round_trip['Transactions'].isna().sum() == daily['Transactions'].isna().sum()
assert round_trip['Lag7'].isna().sum() == daily['Lag7'].isna().sum()
print('PASS CSV dimensions and missingness survive serialization.')

Total observations: 543
Missing branch-days: 0

Missing feature values:
Branch           0
DayOfWeek        0
Month            0
DayOfMonth       0
Weekend          0
Lag1             3
Lag7            21
RollingMean7    21

Missing target values: 0

First 15 rows:
Branch       Date DayOfWeek  Month  DayOfMonth  Weekend  Lag1  Lag7  RollingMean7  Transactions  MissingBranchDay
Garden 2026-01-01  Thursday      1           1        0  <NA>  <NA>           NaN            48             False
Garden 2026-01-02    Friday      1           2        0    48  <NA>           NaN            48             False
Garden 2026-01-03  Saturday      1           3        1    48  <NA>           NaN            36             False
Garden 2026-01-04    Sunday      1           4        1    36  <NA>           NaN            47             False
Garden 2026-01-05    Monday      1           5        0    47  <NA>           NaN            32             False
Garden 2026-01-06   Tuesday      1           6    